# Phase 10 — Training Dynamics

Per-epoch training curves — loss, accuracy/mAP/mIoU, learning-rate schedule, epoch time — for
every run that still has a source to reconstruct them from. See `final_summary.ipynb` for the
macro cross-phase rankings/Pareto/quantization view; this notebook is only about *how training
happened*, not final results.

**Why this is a separate notebook, and from raw logs:** per-epoch history is not kept in any
`*_best.pth` or `*_summary.json` — `ml/reporting.py`'s `make_run_summary` only keeps
epoch-averaged/total numbers, and the one checkpoint format that ever held the full history list
(`*_resume.pth`) doesn't survive past a finished run. Every curve below is reconstructed from a
raw text log or TensorBoard event file instead, via `ml/reporting.py`'s `parse_classification_log`,
`parse_detseg_log`, and `load_tensorboard_scalars`.

**Sources used:**
- Phase 1/3/4 large-scale rerun (`outputs/pcad/archive_legacy_phases/phase_4_5_large_scale/`) — 12 models, FP32+QAT text logs (fetched from PCAD for this notebook).
- Phase 2 partial rerun (`archive_legacy_phases/phase_2_kernel_restriction/`) — 2 models, same format/source.
- Phase 7 detection + segmentation (`outputs/detection_segmentation/phase7/`) — every completed run, text logs.
- Phase 8 CLI-driven models (`outputs/pcad/phase8/`) — 5 models, real TensorBoard event files (richest source).
- Phase 8 notebook-driven models (`checkpoints/phase_8_efficient_vit_hybrid_attention_training/`) — vit_tiny/deit_tiny/qat_vit_tiny, text logs; these are training live as of this data pull, so their curves are partial.
- Phase 9 bypass ablation (`outputs/pcad/phase_9_bypass_ablation/`) — text logs.

**Not recoverable** (checked directly — no `.log`, no non-empty checkpoint history, no wandb run,
on this machine or on PCAD): ~15 other Phase 2/3 kernel-restriction and compensation variants
that were only ever trained in local Jupyter and never rerun elsewhere (`alexnet_2x2_gap`,
`alexnet_2x2_fc`, `alexnet_mixed`, `alexnet_stacked`, `alexnet_groupconv`, `alexnet_factorized`,
`alexnet_residual`, `alexnet_se`, `alexnet_dilated_fc`, `alexnet_dilated_gap`), plus Phase 4's
compression-ablation reruns of existing architectures. Their final numbers still show up in
`final_summary.ipynb`; just not their training curves.

## Configuration

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    """Walk upward until the repository root is found."""
    start = start or Path.cwd()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "models").exists() and (candidate / "results").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))  # so `import ml` resolves regardless of kernel cwd
print(f"Project root: {PROJECT_ROOT}")

RESULTS_DIR = PROJECT_ROOT / "results" / "phase_10_final_summary"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures_generated" / "phase_10_final_summary"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

from ml.plotting import apply_report_style
from ml.reporting import (
    parse_classification_log, parse_detseg_log,
    load_tensorboard_scalars, split_tensorboard_sessions,
)
apply_report_style()

## Shared Plotting Helpers

One small-multiples figure per run, reused across every source below. Classification runs get
loss/accuracy/LR/epoch-time; detection/segmentation runs get loss/metric/LR/epoch-time (the
metric is mAP or mIoU, whichever the log has); TensorBoard runs get the same four panels but
plotted against a chronological write index rather than epoch number — see the Phase 8 section
for why.

In [ ]:
def plot_classification_curves(df: pd.DataFrame, title: str, save_path: Path) -> None:
    if df.empty:
        print(f"[skip] {title}: no epochs parsed")
        return
    fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))

    axes[0].plot(df["epoch"], df["train_loss"], label="train")
    axes[0].plot(df["epoch"], df["val_loss"], label="val")
    axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(fontsize=8)

    axes[1].plot(df["epoch"], df["train_acc"], label="train top-1")
    axes[1].plot(df["epoch"], df["val_acc"], label="val top-1")
    axes[1].plot(df["epoch"], df["val_top5"], label="val top-5", ls="--", alpha=0.7)
    axes[1].set_title("Accuracy (%)"); axes[1].set_xlabel("Epoch"); axes[1].legend(fontsize=8)

    axes[2].plot(df["epoch"], df["lr"], color="tab:green")
    axes[2].set_yscale("log"); axes[2].set_title("LR schedule"); axes[2].set_xlabel("Epoch")

    axes[3].plot(df["epoch"], df["epoch_time_s"], color="tab:orange")
    axes[3].set_title("Epoch time (s)"); axes[3].set_xlabel("Epoch")

    fig.suptitle(title, y=1.04)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()


def plot_detseg_curves(df: pd.DataFrame, title: str, save_path: Path) -> None:
    if df.empty:
        print(f"[skip] {title}: no epochs parsed")
        return
    metric = df["metric_name"].iloc[0]
    fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))

    axes[0].plot(df["epoch"], df["loss"], color="tab:red")
    axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch")

    axes[1].plot(df["epoch"], df["metric_value"], color="tab:blue")
    axes[1].set_title(metric); axes[1].set_xlabel("Epoch")

    axes[2].plot(df["epoch"], df["lr"], color="tab:green")
    axes[2].set_yscale("log"); axes[2].set_title("LR schedule"); axes[2].set_xlabel("Epoch")

    axes[3].plot(df["epoch"], df["epoch_time_s"], color="tab:orange")
    axes[3].set_title("Epoch time (s)"); axes[3].set_xlabel("Epoch")

    fig.suptitle(title, y=1.04)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()


def plot_tensorboard_sessions(df: pd.DataFrame, title: str, save_path: Path) -> None:
    """df is split_tensorboard_sessions's output. Plotted against a per-tag chronological
    write index (not raw `step`) since resumed/requeued runs leave step numbers gapped --
    dotted vertical lines mark where `session` changes (a stage or requeue boundary)."""
    tag_groups = {
        "Loss": ["train_loss", "val_loss"],
        "Accuracy (%)": ["train_acc", "val_acc", "val_top5"],
        "LR schedule": ["lr"],
        "Epoch time (s)": ["epoch_time_s"],
    }
    if df.empty:
        print(f"[skip] {title}: no scalars loaded")
        return
    fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
    for ax, (panel_title, tags) in zip(axes, tag_groups.items()):
        for tag in tags:
            sub = df[df["tag"] == tag].reset_index(drop=True)
            if sub.empty:
                continue
            ax.plot(range(len(sub)), sub["value"], label=tag)
            boundaries = sub.index[sub["session"].diff().fillna(0) != 0]
            for b in boundaries:
                if b > 0:
                    ax.axvline(b, color="grey", ls=":", lw=0.8)
        ax.set_title(panel_title); ax.set_xlabel("Chronological write index")
        if len(tags) > 1:
            ax.legend(fontsize=8)
    axes[2].set_yscale("log")
    fig.suptitle(f"{title} (dotted = stage/requeue boundary)", y=1.04)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()

## 1. Phase 1/3/4 Large-Scale Rerun + Phase 2 Partial Rerun

14 models total (12 large-scale + 2 Phase 2), FP32 and QAT stages where both exist.
`alexnet_final_fire_residual`'s FP32 stage ran across two SLURM submissions (interrupted, then
resumed) and its `Trainer` logging went to `stderr`/SLURM's `.err` capture instead of a `.log`
file — parsed the same way, just concatenated from two files. Its QAT-stage log wasn't found on
PCAD, so it's FP32-only here.

In [ ]:
LARGE_SCALE_DIR = PROJECT_ROOT / "outputs" / "pcad" / "archive_legacy_phases" / "phase_4_5_large_scale"
PHASE2_PARTIAL_DIR = PROJECT_ROOT / "outputs" / "pcad" / "archive_legacy_phases" / "phase_2_kernel_restriction"
FIRE_RESIDUAL_LOGS = [
    PROJECT_ROOT / "outputs" / "pcad" / "logs" / "large_scale" / "train-alexnet_final_fire_residual-802573.err",
    PROJECT_ROOT / "outputs" / "pcad" / "logs" / "large_scale_fire_residual_resume" / "train-alexnet_final_fire_residual-806744.err",
]

# (phase_label, model_name, stage, log_path)
classification_runs = []

for model_dir in sorted(LARGE_SCALE_DIR.iterdir()):
    if not model_dir.is_dir():
        continue
    for log_path in sorted((model_dir / "logs").glob("*.log")):
        stage = "qat" if log_path.stem.startswith("qat_") else "fp32"
        classification_runs.append(("Phase 1/3/4 (large-scale rerun)", model_dir.name, stage, log_path))

for model_dir in sorted(PHASE2_PARTIAL_DIR.iterdir()):
    if not model_dir.is_dir():
        continue
    nested_logs = model_dir / model_dir.name / "logs"  # archive nests an extra model-name dir
    for log_path in sorted(nested_logs.glob("*.log")):
        stage = "qat" if log_path.stem.startswith("qat_") else "fp32"
        classification_runs.append(("Phase 2 (partial rerun)", model_dir.name, stage, log_path))

print(f"{len(classification_runs)} large-scale/Phase-2 logs discovered "
      f"(+1 two-part alexnet_final_fire_residual FP32 run).")

classification_index = []
for phase_label, model_name, stage, log_path in classification_runs:
    df = parse_classification_log(log_path)
    plot_classification_curves(df, f"{model_name} [{stage}] — {phase_label}",
                                FIGURES_DIR / f"dynamics_{model_name}_{stage}.png")
    classification_index.append({
        "phase": phase_label, "model_name": model_name, "stage": stage,
        "source": str(log_path.relative_to(PROJECT_ROOT)), "source_type": "text_log", "n_epochs": len(df),
    })

fire_df = pd.concat([parse_classification_log(p) for p in FIRE_RESIDUAL_LOGS], ignore_index=True)
fire_df = fire_df.drop_duplicates(subset="epoch").sort_values("epoch").reset_index(drop=True)
plot_classification_curves(
    fire_df, "alexnet_final_fire_residual [fp32, two-part resume] — Phase 1/3/4 (large-scale rerun)",
    FIGURES_DIR / "dynamics_alexnet_final_fire_residual_fp32.png",
)
classification_index.append({
    "phase": "Phase 1/3/4 (large-scale rerun)", "model_name": "alexnet_final_fire_residual", "stage": "fp32",
    "source": "outputs/pcad/logs/large_scale*/train-alexnet_final_fire_residual-*.err",
    "source_type": "slurm_stderr", "n_epochs": len(fire_df),
})

## 2. Phase 8 Notebook-Driven Models + Phase 9

`vit_tiny`/`deit_tiny`/`qat_vit_tiny` are training live as of this data pull (no `qat_deit_tiny`
log yet), so their curves stop wherever training currently is — rerun this cell later for more
epochs. Phase 9's two bypass-ablation models use the same log format.

In [ ]:
PHASE8_LOG_DIR = PROJECT_ROOT / "checkpoints" / "phase_8_efficient_vit_hybrid_attention_training"
PHASE9_DIR = PROJECT_ROOT / "outputs" / "pcad" / "phase_9_bypass_ablation"

more_runs = []
for log_path in sorted(PHASE8_LOG_DIR.glob("*.log")):
    stage = "qat" if log_path.stem.startswith("qat_") else "fp32"
    model_name = log_path.stem.removeprefix("qat_")
    more_runs.append(("Phase 8 (notebook-driven)", model_name, stage, log_path))

for log_path in sorted(PHASE9_DIR.glob("**/logs/*.log")):
    stage = "qat" if log_path.stem.startswith("qat_") else "fp32"
    model_name = log_path.stem.removeprefix("qat_")
    more_runs.append(("Phase 9 (bypass ablation)", model_name, stage, log_path))

print(f"{len(more_runs)} Phase 8/9 logs discovered.")

for phase_label, model_name, stage, log_path in more_runs:
    df = parse_classification_log(log_path)
    plot_classification_curves(df, f"{model_name} [{stage}] — {phase_label}",
                                FIGURES_DIR / f"dynamics_{model_name}_{stage}.png")
    classification_index.append({
        "phase": phase_label, "model_name": model_name, "stage": stage,
        "source": str(log_path.relative_to(PROJECT_ROOT)), "source_type": "text_log", "n_epochs": len(df),
    })

## 3. Phase 7 — Detection + Segmentation

Every completed run under `outputs/detection_segmentation/phase7/` (its own per-run log directory,
not the duplicate SLURM-stdout captures under `phase7/logs/`).

In [ ]:
PHASE7_DIR = PROJECT_ROOT / "outputs" / "detection_segmentation" / "phase7"

detseg_index = []
for run_dir in sorted(PHASE7_DIR.iterdir()):
    if not run_dir.is_dir() or run_dir.name == "logs":
        continue
    for log_path in sorted(run_dir.glob("*.log")):
        df = parse_detseg_log(log_path)
        plot_detseg_curves(df, log_path.stem, FIGURES_DIR / f"dynamics_{log_path.stem}.png")
        detseg_index.append({
            "phase": "Phase 7", "model_name": log_path.stem, "stage": None,
            "source": str(log_path.relative_to(PROJECT_ROOT)), "source_type": "text_log", "n_epochs": len(df),
        })

print(f"{len(detseg_index)} Phase 7 detection/segmentation runs plotted.")

## 4. Phase 8 CLI-Driven Models — TensorBoard

`scripts/train.py` opens one `SummaryWriter` per model and reuses it across the FP32 and QAT
stages (and across SLURM requeues), so unlike every other source above, FP32 and QAT land in the
*same* event stream. `split_tensorboard_sessions` marks the boundaries (see its docstring); the
x-axis below is a chronological write index rather than epoch number because step numbers are
gapped across the requeues each of these 5 runs went through.

In [ ]:
PHASE8_TB_DIR = PROJECT_ROOT / "outputs" / "pcad" / "phase8"

tensorboard_index = []
for model_dir in sorted(PHASE8_TB_DIR.iterdir()):
    if not model_dir.is_dir():
        continue
    event_dir = model_dir / "tensorboard" / model_dir.name
    if not event_dir.exists():
        print(f"[skip] {model_dir.name}: no tensorboard dir")
        continue
    raw = load_tensorboard_scalars(event_dir)
    sessions = split_tensorboard_sessions(raw) if not raw.empty else raw
    plot_tensorboard_sessions(sessions, f"{model_dir.name} — Phase 8 (CLI-driven)",
                               FIGURES_DIR / f"dynamics_{model_dir.name}_tensorboard.png")
    tensorboard_index.append({
        "phase": "Phase 8 (CLI-driven)", "model_name": model_dir.name, "stage": "fp32+qat (shared writer)",
        "source": str(event_dir.relative_to(PROJECT_ROOT)), "source_type": "tensorboard",
        "n_epochs": int(raw["step"].nunique()) if not raw.empty else 0,
    })

print(f"{len(tensorboard_index)} Phase 8 CLI-driven models plotted from TensorBoard events.")

## Coverage Summary

In [ ]:
lines = [
    f"- Phase 1/3/4 large-scale rerun + Phase 2 partial rerun: {len(classification_runs) + 1} runs "
    f"across 14 models (alexnet_final_fire_residual is FP32-only -- its QAT log wasn't found on PCAD).",
    f"- Phase 8 notebook-driven + Phase 9: {len(more_runs)} runs "
    f"(vit_tiny/deit_tiny/qat_vit_tiny are training live -- curves are partial as of this data pull).",
    f"- Phase 7 detection/segmentation: {len(detseg_index)} completed runs.",
    f"- Phase 8 CLI-driven: {len(tensorboard_index)}/5 models plotted from TensorBoard events.",
    "- Not recoverable anywhere (checked locally and on PCAD): ~15 Phase 2/3 kernel-restriction "
    "and compensation variants only ever trained in local Jupyter, plus Phase 4's compression-ablation "
    "reruns of existing architectures -- no log, no non-empty checkpoint history, no wandb run.",
]
print("\n".join(lines))

## Persist Results

In [ ]:
training_curves_index = pd.DataFrame(classification_index + detseg_index + tensorboard_index)
out_path = RESULTS_DIR / "training_curves_index.csv"
training_curves_index.to_csv(out_path, index=False)
print(f"Saved: {out_path} ({len(training_curves_index)} runs indexed)")
display(training_curves_index)